
# Products Bronze — Hardened ingestion project

Supports `INCREMENTAL`, `BACKFILL`, and `FULL_LOAD` for `products_YYYYMMDD.json` files.

- Explicit source schema (`PRODUCTS_SCHEMA`) is applied to every JSON read.
- Discovery, SHA-256 hashing, reconciliation, selection, and preflight are read-only.
- `dry_run=true` is the default and prevents manifest/Bronze writes.
- `dry_run=false` enables registration of unregistered Products files, conditional manifest claims, file-level validation, targeted Bronze replacement, and manifest finalization.
- Uses only the Products manifest records and Products Bronze path. It does not intentionally update Orders or the isolated test manifest.

**Operational limits:** use one writer for this dataset. Bronze Delta writes and manifest Delta writes are separate transactions, not one atomic transaction. If a result says `FAILED_AFTER_BRONZE_WRITE_RECONCILIATION_REQUIRED`, inspect the manifest and Bronze for that file before retrying. Validate the first write run in a non-production/test copy if available.


In [0]:

# CELL 1 — CONFIGURATION AND WIDGETS
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, DoubleType
)
from datetime import datetime, timezone, timedelta
import uuid

DATASET_NAME = "products"
SOURCE_SYSTEM = "product_api"
STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
LANDING_PATH = f"{BASE_PATH}landing/{DATASET_NAME}/"
BRONZE_PATH = f"{BASE_PATH}bronze/{DATASET_NAME}/"
MANIFEST_PATH = f"{BASE_PATH}metadata/file_manifest/"
LEASE_MINUTES = 60

PRODUCTS_SCHEMA = StructType([
    StructField("product_id", LongType(), nullable=False),
    StructField("product_name", StringType(), nullable=True),
    StructField("category", StringType(), nullable=True),
    StructField("brand", StringType(), nullable=True),
    StructField("price", DoubleType(), nullable=True),
])

for name in ["run_mode", "backfill_start_date", "backfill_end_date", "reprocess_successful", "dry_run"]:
    try:
        dbutils.widgets.remove(name)
    except Exception:
        pass

dbutils.widgets.dropdown("run_mode", "INCREMENTAL", ["INCREMENTAL", "BACKFILL", "FULL_LOAD"], "Run Mode")
dbutils.widgets.text("backfill_start_date", "", "Backfill Start Date (YYYY-MM-DD)")
dbutils.widgets.text("backfill_end_date", "", "Backfill End Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("reprocess_successful", "false", ["false", "true"], "Reprocess Successful Files?")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "Dry Run? (true = no writes)")

RUN_MODE = dbutils.widgets.get("run_mode").strip().upper()
BACKFILL_START_DATE = dbutils.widgets.get("backfill_start_date").strip()
BACKFILL_END_DATE = dbutils.widgets.get("backfill_end_date").strip()
REPROCESS_SUCCESSFUL = dbutils.widgets.get("reprocess_successful").strip().lower() == "true"
DRY_RUN = dbutils.widgets.get("dry_run").strip().lower() == "true"

assert RUN_MODE in {"INCREMENTAL", "BACKFILL", "FULL_LOAD"}
if RUN_MODE == "BACKFILL":
    assert BACKFILL_START_DATE and BACKFILL_END_DATE, "BACKFILL requires both dates."
    start_date = datetime.strptime(BACKFILL_START_DATE, "%Y-%m-%d").date()
    end_date = datetime.strptime(BACKFILL_END_DATE, "%Y-%m-%d").date()
    assert start_date <= end_date, "Backfill start date must be <= end date."

print("Mode:", RUN_MODE, "| dry_run:", DRY_RUN, "| reprocess_successful:", REPROCESS_SUCCESSFUL)
print("Landing:", LANDING_PATH)
print("Bronze:", BRONZE_PATH)
print("Manifest:", MANIFEST_PATH)


Mode: FULL_LOAD | dry_run: False | reprocess_successful: True
Landing: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/
Bronze: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/
Manifest: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/


In [0]:

# CELL 2 — DISCOVER FILES AND CALCULATE SHA-256 (READ-ONLY)
def list_files_recursive(path):
    found = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            found.extend(list_files_recursive(item.path))
        else:
            found.append((item.name, item.path, int(item.size)))
    return found

discovered = list_files_recursive(LANDING_PATH)
landing_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_size_bytes", LongType(), False),
])
landing_df = spark.createDataFrame(discovered, landing_schema) if discovered else spark.createDataFrame([], landing_schema)

dated_df = (
    landing_df
    .withColumn("date_string", F.regexp_extract("file_name", r"^products_(\d{8})\.json$", 1))
    .withColumn("data_date", F.to_date("date_string", "yyyyMMdd"))
    .withColumn("filename_status", F.when(F.col("date_string") == "", "INVALID_FILENAME")
                .when(F.col("data_date").isNull(), "INVALID_DATE").otherwise("VALID"))
)

invalid = dated_df.filter(F.col("filename_status") != "VALID").count()
assert invalid == 0, "Invalid file names/dates found. Fix them before continuing."

binary_df = (
    spark.read.format("binaryFile").load(LANDING_PATH + "**")
    .select(F.col("path").alias("file_path"), F.sha2("content", 256).alias("actual_content_hash"))
)
files_df = dated_df.filter(F.col("filename_status") == "VALID").join(binary_df, "file_path", "left")
assert files_df.filter(F.col("actual_content_hash").isNull()).count() == 0, "A file could not be hashed."
display(files_df.select("file_name", "data_date", "file_size_bytes", "actual_content_hash").orderBy("data_date"))
print("Discovered files:", files_df.count())


file_name,data_date,file_size_bytes,actual_content_hash
products_20261007.json,2026-10-07,3662500,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
products_20261008.json,2026-10-08,2615396,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b


Discovered files: 2


In [0]:

# CELL 3 — RECONCILE AND SELECT FILES (READ-ONLY)
manifest_all_df = spark.read.format("delta").load(MANIFEST_PATH)
manifest_df = (
    manifest_all_df.filter(F.col("dataset") == DATASET_NAME)
    .select("dataset", "file_name", "status", "content_hash", "data_date",
            "retry_count", "attempt_count", "run_id")
)
duplicates = manifest_df.groupBy("dataset", "file_name").count().filter("count > 1").count()
assert duplicates == 0, "Duplicate Products manifest keys found."

reconciliation_df = (
    files_df.alias("l")
    .join(manifest_df.alias("m"), F.col("l.file_name") == F.col("m.file_name"), "left")
    .select(
        F.col("l.file_name"), F.col("l.file_path"), F.col("l.file_size_bytes"),
        F.col("l.data_date").alias("landing_data_date"), F.col("l.actual_content_hash"),
        F.col("m.status").alias("manifest_status"), F.col("m.content_hash").alias("manifest_hash"),
        F.col("m.data_date").alias("manifest_data_date")
    )
    .withColumn("effective_data_date", F.coalesce("manifest_data_date", "landing_data_date"))
    .withColumn(
        "selection_reason",
        F.when(F.col("manifest_status") == "PROCESSING", "SKIP_PROCESSING")
         .when(F.col("manifest_status").isNull(), "REGISTER_REQUIRED")
         .when(F.col("manifest_hash").isNull(), "REVIEW_REQUIRED")
         .when(F.col("manifest_hash") != F.col("actual_content_hash"), "CONTENT_CHANGED")
         .when(F.col("manifest_status").isin("NEW", "FAILED"), "ELIGIBLE")
         .when(F.col("manifest_status") == "SUCCESS", "SUCCESS_REPROCESS_CANDIDATE")
         .otherwise("REVIEW_REQUIRED")
    )
)

base_reasons = ["REGISTER_REQUIRED", "ELIGIBLE", "CONTENT_CHANGED"]
if RUN_MODE == "INCREMENTAL":
    selected_df = reconciliation_df.filter(F.col("selection_reason").isin(base_reasons))
elif RUN_MODE == "BACKFILL":
    reasons = base_reasons + (["SUCCESS_REPROCESS_CANDIDATE"] if REPROCESS_SUCCESSFUL else [])
    selected_df = reconciliation_df.filter(
        F.col("effective_data_date").between(
            F.to_date(F.lit(BACKFILL_START_DATE)), F.to_date(F.lit(BACKFILL_END_DATE))
        ) & F.col("selection_reason").isin(reasons)
    )
else:  # FULL_LOAD includes successful files regardless of reprocess_successful
    selected_df = reconciliation_df.filter(
        F.col("selection_reason").isin(base_reasons + ["SUCCESS_REPROCESS_CANDIDATE"])
    )

display(reconciliation_df.select("file_name", "effective_data_date", "manifest_status",
                                  "selection_reason").orderBy("effective_data_date"))
print("Selected candidates:", selected_df.count())


file_name,effective_data_date,manifest_status,selection_reason
products_20261007.json,2026-10-07,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
products_20261008.json,2026-10-08,SUCCESS,SUCCESS_REPROCESS_CANDIDATE


Selected candidates: 2


In [0]:

# CELL 4 — PREFLIGHT, SAFETY GATE, AND DRY-RUN PLAN (READ-ONLY)
manifest_required = {
    "dataset", "file_name", "file_path", "status", "first_seen", "processed_at",
    "run_id", "retry_count", "error_message", "data_date", "file_size_bytes",
    "content_hash", "last_seen_at", "discovery_run_id", "claimed_at",
    "lease_expires_at", "last_attempt_at", "attempt_count", "last_success_at",
    "status_updated_at", "source_record_count", "written_record_count",
    "rejected_record_count", "processed_content_hash", "target_path",
    "target_delta_version", "processing_duration_seconds", "error_type", "last_error_at"
}
manifest_missing = sorted(manifest_required - set(manifest_all_df.columns))
assert not manifest_missing, f"Manifest schema is missing required columns: {manifest_missing}"

bronze_required = {
    "product_id", "product_name", "category", "brand", "price",
    "_source_file", "_source_path", "_dataset", "_batch_id",
    "_run_id", "_ingested_at", "_source_system"
}
bronze_df_existing = spark.read.format("delta").load(BRONZE_PATH)
bronze_missing = sorted(bronze_required - set(bronze_df_existing.columns))
assert not bronze_missing, f"Bronze schema is missing required columns: {bronze_missing}"

selected_count = selected_df.count()
duplicate_selected = selected_df.groupBy("file_name").count().filter("count > 1").count()
bad_hashes = selected_df.filter(
    F.col("actual_content_hash").isNull() | (F.length("actual_content_hash") != 64)
).count()
bad_dates = selected_df.filter(F.col("effective_data_date").isNull()).count()
processing_selected = selected_df.filter(F.col("manifest_status") == "PROCESSING").count()
review_selected = selected_df.filter(F.col("selection_reason") == "REVIEW_REQUIRED").count()

assert duplicate_selected == 0, "Duplicate selected file names found."
assert bad_hashes == 0, "A selected file has an invalid/missing SHA-256 hash."
assert bad_dates == 0, "A selected file has no effective data date."
assert processing_selected == 0, "A selected file is already PROCESSING."
assert review_selected == 0, "A selected file requires manual review."

display(selected_df.select(
    "file_name", "file_path", "effective_data_date",
    "manifest_status", "selection_reason", "actual_content_hash"
).orderBy("effective_data_date", "file_name"))

print("Preflight PASS")
print("Selected candidates:", selected_count)
print("Manifest and Bronze required columns: PASS")
print("Dry run:", DRY_RUN)
if DRY_RUN:
    print("DRY RUN: no manifest or Bronze writes will occur.")


file_name,file_path,effective_data_date,manifest_status,selection_reason,actual_content_hash
products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,2026-10-07,SUCCESS,SUCCESS_REPROCESS_CANDIDATE,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
products_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json,2026-10-08,SUCCESS,SUCCESS_REPROCESS_CANDIDATE,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b


Preflight PASS
Selected candidates: 2
Manifest and Bronze required columns: PASS
Dry run: False


In [0]:

# CELL 5 — PROCESS SELECTED FILES
# Writes only when dry_run=false. Use ONE writer for this dataset.
if DRY_RUN:
    print("DRY RUN complete. No manifest or Bronze writes performed.")
else:
    run_id = str(uuid.uuid4())
    print("Processing run:", run_id)

    manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)
    manifest_schema = spark.read.format("delta").load(MANIFEST_PATH).schema
    manifest_columns = [f.name for f in manifest_schema.fields]
    results = []

    def make_manifest_row(file_name, file_path, file_size, data_date, content_hash, discovery_run_id):
        now_naive = datetime.now(timezone.utc).replace(tzinfo=None)
        values = {name: None for name in manifest_columns}
        values.update({
            "dataset": DATASET_NAME,
            "file_name": file_name,
            "file_path": file_path,
            "status": "NEW",
            "first_seen": now_naive,
            "retry_count": 0,
            "data_date": data_date,
            "file_size_bytes": int(file_size),
            "content_hash": content_hash,
            "last_seen_at": now_naive,
            "discovery_run_id": discovery_run_id,
            "attempt_count": 0,
            "status_updated_at": now_naive,
        })
        # Construct with the exact existing manifest schema to preserve column types.
        return spark.createDataFrame([values], schema=manifest_schema)

    for candidate in selected_df.orderBy("effective_data_date", "file_name").collect():
        file_name = candidate["file_name"]
        file_path = candidate["file_path"]
        actual_hash = candidate["actual_content_hash"]
        selection_reason = candidate["selection_reason"]
        bronze_write_committed = False
        claim_acquired = False

        try:
            # Re-check source content immediately before any manifest mutation.
            fresh_hash = (
                spark.read.format("binaryFile").load(file_path)
                .select(F.sha2("content", 256).alias("h")).first()["h"]
            )
            assert fresh_hash == actual_hash, f"{file_name}: source changed after discovery."

            current_rows = (
                spark.read.format("delta").load(MANIFEST_PATH)
                .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                .select("status", "content_hash", "attempt_count", "retry_count")
                .collect()
            )

            # Register new files only through a key-based MERGE into the global manifest.
            if not current_rows:
                assert selection_reason == "REGISTER_REQUIRED", (
                    f"{file_name}: manifest row disappeared after selection; rerun discovery."
                )
                row = make_manifest_row(
                    file_name, file_path, candidate["file_size_bytes"],
                    candidate["effective_data_date"], actual_hash, run_id
                )
                (manifest_delta.alias("t")
                    .merge(row.alias("s"),
                           "t.dataset = s.dataset AND t.file_name = s.file_name")
                    .whenNotMatchedInsertAll()
                    .execute())
                current_rows = (
                    spark.read.format("delta").load(MANIFEST_PATH)
                    .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                    .select("status", "content_hash", "attempt_count", "retry_count")
                    .collect()
                )

            assert len(current_rows) == 1, f"{file_name}: expected exactly one manifest row after registration."
            current = current_rows[0]
            expected_status = current["status"]
            expected_hash = current["content_hash"]
            assert expected_status in {"NEW", "FAILED", "SUCCESS"}, (
                f"{file_name}: status {expected_status!r} is not eligible for processing."
            )
            if selection_reason == "CONTENT_CHANGED":
                assert expected_hash is not None and expected_hash != actual_hash, (
                    f"{file_name}: expected a genuine content-hash change; rerun discovery."
                )
            else:
                assert expected_hash == actual_hash, (
                    f"{file_name}: manifest hash differs from discovered source hash; rerun discovery/reconciliation."
                )
            if selection_reason == "SUCCESS_REPROCESS_CANDIDATE":
                if RUN_MODE == "INCREMENTAL":
                    raise RuntimeError(f"{file_name}: successful-file reprocessing is not allowed in INCREMENTAL mode.")
                if RUN_MODE == "BACKFILL" and not REPROCESS_SUCCESSFUL:
                    raise RuntimeError(f"{file_name}: successful-file reprocessing is disabled for this backfill.")

            now_naive = datetime.now(timezone.utc).replace(tzinfo=None)
            lease_expiry = now_naive + timedelta(minutes=LEASE_MINUTES)

            # Conditional claim. One-writer operation is required; read-back verifies ownership.
            claim_condition = (
                (F.col("t.dataset") == DATASET_NAME) &
                (F.col("t.file_name") == file_name) &
                (F.col("t.status") == expected_status) &
                (F.col("t.content_hash") == expected_hash)
            )
            manifest_delta.alias("t").update(
                condition=claim_condition,
                set={
                    "status": F.lit("PROCESSING"),
                    "run_id": F.lit(run_id),
                    "claimed_at": F.lit(now_naive).cast("timestamp"),
                    "lease_expires_at": F.lit(lease_expiry).cast("timestamp"),
                    "last_attempt_at": F.lit(now_naive).cast("timestamp"),
                    "attempt_count": F.coalesce(F.col("t.attempt_count"), F.lit(0)) + F.lit(1),
                    "status_updated_at": F.lit(now_naive).cast("timestamp"),
                    "content_hash": F.lit(actual_hash),
                    "error_message": F.lit(None).cast("string"),
                    "error_type": F.lit(None).cast("string"),
                }
            )

            owner_rows = (
                spark.read.format("delta").load(MANIFEST_PATH)
                .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                .select("status", "run_id", "content_hash")
                .collect()
            )
            assert len(owner_rows) == 1, f"{file_name}: manifest row missing after claim."
            owner = owner_rows[0]
            assert owner["status"] == "PROCESSING" and owner["run_id"] == run_id, (
                f"{file_name}: claim was not acquired by this run."
            )
            assert owner["content_hash"] == actual_hash, f"{file_name}: claimed hash mismatch."
            claim_acquired = True

            source_df = (
                spark.read.schema(PRODUCTS_SCHEMA)
                .option("multiLine", "true")
                .option("mode", "FAILFAST")
                .json(file_path)
            )
            source_count = source_df.count()
            null_ids = source_df.filter(F.col("product_id").isNull()).count()
            distinct_ids = source_df.select("product_id").distinct().count()
            negative_prices = source_df.filter(F.col("price") < 0).count()

            assert source_count > 0, f"{file_name}: source is empty."
            assert null_ids == 0, f"{file_name}: null product IDs found."
            assert distinct_ids == source_count, f"{file_name}: duplicate product IDs found within this file."
            assert negative_prices == 0, f"{file_name}: negative prices found."

            ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)
            batch_id = str(uuid.uuid4())
            replacement_df = (
                source_df
                .withColumn("_source_file", F.lit(file_name))
                .withColumn("_source_path", F.lit(file_path))
                .withColumn("_dataset", F.lit(DATASET_NAME))
                .withColumn("_batch_id", F.lit(batch_id))
                .withColumn("_run_id", F.lit(run_id))
                .withColumn("_ingested_at", F.lit(ingested_at).cast("timestamp"))
                .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
            )

            # Verify target Delta schema is compatible before writing.
            bronze_before_df = spark.read.format("delta").load(BRONZE_PATH)
            bronze_columns = set(bronze_before_df.columns)
            assert bronze_required.issubset(bronze_columns), "Bronze schema changed after preflight."
            target_before = bronze_before_df.filter(F.col("_source_file") == file_name).count()

            # Replace only the current file's rows; other source files are preserved.
            (replacement_df.write.format("delta").mode("overwrite")
                .option("replaceWhere", f"_source_file = '{file_name}'")
                .save(BRONZE_PATH))
            bronze_write_committed = True

            post_target = (
                spark.read.format("delta").load(BRONZE_PATH)
                .filter(F.col("_source_file") == file_name)
            )
            written_count = post_target.count()
            post_nulls = post_target.filter(F.col("product_id").isNull()).count()
            wrong_run = post_target.filter(F.col("_run_id") != run_id).count()
            wrong_batch = post_target.filter(F.col("_batch_id") != batch_id).count()
            assert written_count == source_count, f"{file_name}: written count mismatch."
            assert post_nulls == 0 and wrong_run == 0 and wrong_batch == 0, (
                f"{file_name}: post-write validation failed."
            )

            version = DeltaTable.forPath(spark, BRONZE_PATH).history(1).select("version").first()["version"]
            finalized_at = datetime.now(timezone.utc).replace(tzinfo=None)
            manifest_delta.alias("t").update(
                condition=(
                    (F.col("t.dataset") == DATASET_NAME) &
                    (F.col("t.file_name") == file_name) &
                    (F.col("t.status") == "PROCESSING") &
                    (F.col("t.run_id") == run_id) &
                    (F.col("t.content_hash") == actual_hash)
                ),
                set={
                    "status": F.lit("SUCCESS"),
                    "batch_id": F.lit(batch_id),
                    "processed_at": F.lit(finalized_at).cast("timestamp"),
                    "last_success_at": F.lit(finalized_at).cast("timestamp"),
                    "status_updated_at": F.lit(finalized_at).cast("timestamp"),
                    "processed_content_hash": F.lit(actual_hash),
                    "source_record_count": F.lit(source_count).cast("long"),
                    "written_record_count": F.lit(written_count).cast("long"),
                    "rejected_record_count": F.lit(0).cast("long"),
                    "target_path": F.lit(BRONZE_PATH),
                    "target_delta_version": F.lit(version).cast("long"),
                    "error_message": F.lit(None).cast("string"),
                    "error_type": F.lit(None).cast("string"),
                    "last_error_at": F.lit(None).cast("timestamp"),
                    "lease_expires_at": F.lit(None).cast("timestamp"),
                }
            )

            final_rows = (
                spark.read.format("delta").load(MANIFEST_PATH)
                .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                .select("status", "run_id", "written_record_count", "target_delta_version")
                .collect()
            )
            assert len(final_rows) == 1, f"{file_name}: final manifest row missing."
            final_row = final_rows[0]
            assert final_row["status"] == "SUCCESS" and final_row["run_id"] == run_id, (
                f"{file_name}: finalization did not confirm SUCCESS for this run."
            )
            assert final_row["written_record_count"] == written_count, f"{file_name}: final count mismatch."

            results.append((file_name, "SUCCESS", source_count, written_count, version, ""))
            print(
                f"SUCCESS {file_name}: source={source_count}, written={written_count}, "
                f"prior_target={target_before}, Delta version={version}"
            )

        except Exception as exc:
            failed_at = datetime.now(timezone.utc).replace(tzinfo=None)
            error_text = str(exc)[:4000]
            # If Bronze was committed but finalization/verification failed, record this explicitly.
            # This is not an atomic transaction across Bronze and manifest; reconcile before retrying.
            if claim_acquired:
                try:
                    current_owner = (
                        spark.read.format("delta").load(MANIFEST_PATH)
                        .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                        .select("status", "run_id")
                        .first()
                    )
                    if current_owner and current_owner["status"] == "PROCESSING" and current_owner["run_id"] == run_id:
                        if bronze_write_committed:
                            error_text = (
                                "BRONZE_WRITE_COMMITTED_BUT_FINALIZATION_FAILED; reconcile Bronze and manifest before retry. "
                                + error_text
                            )[:4000]
                        manifest_delta.alias("t").update(
                            condition=(
                                (F.col("t.dataset") == DATASET_NAME) &
                                (F.col("t.file_name") == file_name) &
                                (F.col("t.status") == "PROCESSING") &
                                (F.col("t.run_id") == run_id)
                            ),
                            set={
                                "status": F.lit("FAILED"),
                                "status_updated_at": F.lit(failed_at).cast("timestamp"),
                                "last_error_at": F.lit(failed_at).cast("timestamp"),
                                "error_message": F.lit(error_text),
                                "error_type": F.lit(type(exc).__name__),
                                "lease_expires_at": F.lit(None).cast("timestamp"),
                            }
                        )
                except Exception as manifest_exc:
                    print(f"WARNING: could not update failure state for {file_name}: {manifest_exc}")

            state = "FAILED_AFTER_BRONZE_WRITE_RECONCILIATION_REQUIRED" if bronze_write_committed else "FAILED"
            print(f"{state} {file_name}: {type(exc).__name__}: {exc}")
            results.append((file_name, state, None, None, None, error_text))

    result_schema = (
        "file_name string, status string, source_count long, "
        "written_count long, delta_version long, error_message string"
    )
    results_df = spark.createDataFrame(results, result_schema)
    display(results_df)
    print("Run ID:", run_id)
    print("Important: if any row reports FAILED_AFTER_BRONZE_WRITE_RECONCILIATION_REQUIRED, reconcile that file before retrying.")


Processing run: cdd59516-7d65-4ee1-90e3-1fa95d613e28
SUCCESS products_20261007.json: source=25000, written=25000, prior_target=25000, Delta version=4
SUCCESS products_20261008.json: source=25000, written=25000, prior_target=25000, Delta version=5


file_name,status,source_count,written_count,delta_version,error_message
products_20261007.json,SUCCESS,25000,25000,4,
products_20261008.json,SUCCESS,25000,25000,5,


Run ID: cdd59516-7d65-4ee1-90e3-1fa95d613e28
Important: if any row reports FAILED_AFTER_BRONZE_WRITE_RECONCILIATION_REQUIRED, reconcile that file before retrying.


In [0]:

# CELL 6 — FINAL INVENTORY AND MANIFEST CHECK (READ-ONLY)
bronze_final_df = spark.read.format("delta").load(BRONZE_PATH)
display(
    bronze_final_df.groupBy("_source_file").agg(
        F.count("*").alias("record_count"),
        F.countDistinct("product_id").alias("distinct_product_ids"),
        F.sum(F.when(F.col("product_id").isNull(), 1).otherwise(0)).alias("null_product_ids")
    ).orderBy("_source_file")
)

display(
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(F.col("dataset") == DATASET_NAME)
    .select("file_name", "status", "source_record_count", "written_record_count",
            "processed_content_hash", "target_delta_version", "lease_expires_at")
    .orderBy("file_name")
)
print("Total Bronze rows:", bronze_final_df.count())


_source_file,record_count,distinct_product_ids,null_product_ids
products_20261007.json,25000,25000,0
products_20261008.json,25000,25000,0


file_name,status,source_record_count,written_record_count,processed_content_hash,target_delta_version,lease_expires_at
products_20261007.json,SUCCESS,25000,25000,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,4,null
products_20261008.json,SUCCESS,25000,25000,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,5,null


Total Bronze rows: 50000
